# Notebook 05 — Cross-source synthesis

**Purpose:** First cross-page reasoning step. Read N existing `SourcePage`s and produce ONE new `ConceptPage` (or `AnalysisPage` / `DecisionPage`) that cites all of them via `[[wikilink]]` references.

**Exam relevance:** Prompt Engineering (XML, CoT) + Architecture Patterns (RAG-vs-compounding-wiki).
**Design refs:** `docs/marginalia-design.md` §7.1 (synthesis agent), §10 Scenario A step 5, §7.3.1 (strict-schema retry + dangling-link enforcement).
**Depends on:** NB 02 (analyze/synthesize pipeline), NB 04 (multi-modal ingest produces the SourcePage fixtures).

**Fixtures** (all under `data/poc-wiki/sources/`):

- 3 *auto-generated* SourcePages (regenerate via `uv run python notebooks/_ops/build_source_pages.py`):
  - `apollo-launch-blockers-sync.md` (text path)
  - `marginalia-engine-architecture-and-model-selection.md` (PDF text path)
  - `marginalia-ingest-pipeline-architecture.md` (image vision path)
- 2 *hand-curated* contradictory SourcePages for the edge case:
  - `apollo-q2.md` (Apollo launches Q2 2026)
  - `apollo-q3.md` (Apollo postponed to Q3 2026)


In [ ]:
# Enable autoreload so edits to engine modules are picked up automatically.
%load_ext autoreload
%autoreload 2

import os
import json
import time
from pathlib import Path
from datetime import date

from dotenv import load_dotenv
from anthropic import Anthropic
import frontmatter

from engine.agents.synthesis import (
    synthesize_cross_source,
    extract_wikilinks,
    derive_default_path,
)
from engine.models.pages import (
    SourcePage,
    PageStatus,
    PageType,
    ConceptPage,
    AnalysisPage,
)
from engine.models.wiki_config import MarginaliaConfig
from engine.utils.cost_tracker import estimate_cost_usd, record_attempt


In [ ]:
load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not in env"

client = Anthropic()
config = MarginaliaConfig.load(Path("data/poc-wiki"))
print(f"config loaded: purpose={len(config.purpose_body)} chars, agents={len(config.agents_body)} chars")


## Cell 3 — Load committed SourcePage fixtures

Read three pre-ingested SourcePages off disk via `python-frontmatter`,
parse them into Pydantic objects. These are the inputs to cross-source
synthesis. Each was produced by NB 02/04 ingest on a different
extraction method (text, PDF document block, image vision).

In [ ]:
SOURCES_DIR = Path("data/poc-wiki/sources")

CORE_FIXTURES = [
    "apollo-launch-blockers-sync.md",
    "marginalia-engine-architecture-and-model-selection.md",
    "marginalia-ingest-pipeline-architecture.md",
]


def _load_source_page(path: Path) -> tuple[SourcePage, str]:
    post = frontmatter.load(path)
    return SourcePage.model_validate(post.metadata), post.content


sources: list[SourcePage] = []
bodies: list[str] = []
for name in CORE_FIXTURES:
    path = SOURCES_DIR / name
    page, body = _load_source_page(path)
    sources.append(page)
    bodies.append(body)
    print(f"loaded {name}: {page.title!r}")

# Compute derived paths (the wikilinks the synthesizer must cite).
source_paths = [derive_default_path(s) for s in sources]
print()
for s, p in zip(sources, source_paths):
    print(f"  [[{p}]] ← {s.title}")


## Cell 4 — Vanilla cross-source synthesis

The default Concept synthesis: 3 sources → 1 ConceptPage citing all
three. Strict-schema retry + dangling-citation verification all run
inside `synthesize_cross_source`.

In [ ]:
import yaml
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax

t0 = time.monotonic()
vanilla_page, vanilla_body, vanilla_log = await synthesize_cross_source(
    sources, config, client=client
)
vanilla_wall = time.monotonic() - t0
print(f"status={vanilla_page.status.value}  attempts={len(vanilla_log)}  wall={vanilla_wall:.2f}s")

console = Console()
fm_yaml = yaml.safe_dump(vanilla_page.model_dump(mode="json", exclude_none=True), sort_keys=False).rstrip()
console.print(
    Panel(
        Syntax(f"---\n{fm_yaml}\n---", "yaml", theme="ansi_dark", background_color="default"),
        title=f"vanilla concept ({vanilla_page.title})",
        border_style="green",
    )
)
print()
print("--- body ---")
print(vanilla_body)


## Cell 5 — Citation verification visualizer

The retry loop already enforced this; the cell renders the resolution
state for human eyeballing. Each `[[wikilink]]` in the rendered page
is checked against `available_paths` (set of input source paths).

In [ ]:
from rich.table import Table


def _all_wikilinks(page, body):
    fm = page.model_dump(mode="json", exclude_none=True)
    found = set()
    for field in ("related", "contradicts"):
        for v in fm.get(field) or []:
            found.update(extract_wikilinks(v) if isinstance(v, str) else [])
    if isinstance(fm.get("supersedes"), str):
        found.update(extract_wikilinks(fm["supersedes"]))
    found.update(extract_wikilinks(body))
    return sorted(found)


available = set(source_paths)
links = _all_wikilinks(vanilla_page, vanilla_body)

cite_table = Table(title="Citation resolution — vanilla synthesis")
cite_table.add_column("wikilink")
cite_table.add_column("resolves to")
cite_table.add_column("status")

for link in links:
    if link in available:
        cite_table.add_row(f"[[{link}]]", link, "[green]resolved[/green]")
    else:
        cite_table.add_row(f"[[{link}]]", "—", "[red]dangling[/red]")

# Coverage: did we cite every input?
cite_table.add_section()
for path in source_paths:
    if path in links:
        cite_table.add_row(f"[[{path}]] (input)", path, "[green]cited[/green]")
    else:
        cite_table.add_row(f"[[{path}]] (input)", path, "[yellow]uncited[/yellow]")

console.print(cite_table)


## Cell 6 — Hint-driven variant

Same sources, `hint="frame this around the platform-engineering theme"`.
Watch how the same inputs produce a differently-framed output: title,
summary, body emphasis. The hint is not a hard constraint — it's a
soft steer that the model can choose how to honor.

In [ ]:
t0 = time.monotonic()
hint_page, hint_body, hint_log = await synthesize_cross_source(
    sources,
    config,
    hint="frame this around the platform-engineering theme",
    client=client,
)
hint_wall = time.monotonic() - t0
print(f"status={hint_page.status.value}  attempts={len(hint_log)}  wall={hint_wall:.2f}s")

console.print(
    Panel(
        f"[bold]vanilla:[/bold] {vanilla_page.title}\n[bold]hint:[/bold] {hint_page.title}",
        title="title shift",
        border_style="cyan",
    )
)
print("\n--- vanilla body (first 200 chars) ---")
print(vanilla_body[:200])
print("\n--- hint body (first 200 chars) ---")
print(hint_body[:200])


## Cell 7 — Target-type variant: AnalysisPage

Same sources, `target_type=PageType.ANALYSIS`. The synthesizer
re-casts the page: required `confidence` and `sources` (≥1 SourceRef)
fields are populated; the discriminator becomes `analysis`.

In [ ]:
t0 = time.monotonic()
analysis_page, analysis_body, analysis_log = await synthesize_cross_source(
    sources,
    config,
    target_type=PageType.ANALYSIS,
    client=client,
)
analysis_wall = time.monotonic() - t0
print(f"status={analysis_page.status.value}  attempts={len(analysis_log)}  wall={analysis_wall:.2f}s")
print(f"page type = {analysis_page.type.value}")
print(f"required `sources` count: {len(analysis_page.sources)}")
print(f"required `confidence`: {analysis_page.confidence}")
print()
print("--- analysis body (first 300 chars) ---")
print(analysis_body[:300])


## Cell 8 — Edge case: contradictory sources

Two hand-curated SourcePages disagree: one says Apollo launches Q2
2026, the other says it's postponed to Q3. Run synthesis and observe
how Sonnet handles the conflict.

Expected behaviors (any one is reasonable):
1. Populate `contradicts: [[...]]` on the new page and discuss the
   conflict in the body.
2. Pick one source and note the other as superseded in the body.
3. Fall back to draft if the contradiction can't be reconciled within
   the schema.

In [ ]:
# Load the contradictory pair.
q2_page, q2_body = _load_source_page(SOURCES_DIR / "apollo-q2.md")
q3_page, q3_body = _load_source_page(SOURCES_DIR / "apollo-q3.md")
contradictory_sources = [q2_page, q3_page]

t0 = time.monotonic()
conflict_page, conflict_body, conflict_log = await synthesize_cross_source(
    contradictory_sources, config, client=client
)
conflict_wall = time.monotonic() - t0

fm = conflict_page.model_dump(mode="json", exclude_none=True)
print(f"status   = {conflict_page.status.value}")
print(f"attempts = {len(conflict_log)}")
print(f"wall     = {conflict_wall:.2f}s")
print(f"contradicts populated? {bool(fm.get('contradicts'))}")
print(f"contradicts = {fm.get('contradicts', [])}")
print()
print("--- body (first 500 chars) ---")
print(conflict_body[:500])


## Cell 9 — Cost + attempt summary

In [ ]:
def _cost_for(log):
    return sum(
        estimate_cost_usd("claude-sonnet-4-6", e["input_tokens"], e["output_tokens"])
        for e in log
    )

cost_table = Table(title="Cross-source synthesis — cost summary")
cost_table.add_column("variant")
cost_table.add_column("attempts", justify="right")
cost_table.add_column("tokens (in/out)", justify="right")
cost_table.add_column("cost (USD)", justify="right")
cost_table.add_column("wall (s)", justify="right")

for label, log, wall in [
    ("vanilla concept",    vanilla_log,  vanilla_wall),
    ("hinted concept",     hint_log,     hint_wall),
    ("analysis target",    analysis_log, analysis_wall),
    ("contradictory pair", conflict_log, conflict_wall),
]:
    tin = sum(e["input_tokens"] for e in log)
    tout = sum(e["output_tokens"] for e in log)
    cost_table.add_row(
        label,
        str(len(log)),
        f"{tin}/{tout}",
        f"${_cost_for(log):.6f}",
        f"{wall:.2f}",
    )

console.print(cost_table)


## Cell 10 — Receipts: write `engine/decisions/synthesis-patterns.md`

Parallel to NB 03's `model-selection.md` and NB 04's
`multimodal-dispatch.md`. Captures cross-source synthesis behavior so
future work can re-evaluate when prompts, models, or fixtures shift.

In [ ]:
DECISIONS_PATH = Path("../engine/decisions/synthesis-patterns.md")
DECISIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

def _attempt_summary(log):
    tin = sum(e["input_tokens"] for e in log)
    tout = sum(e["output_tokens"] for e in log)
    return len(log), tin, tout, _cost_for(log)

vanilla_attempts, vanilla_in, vanilla_out, vanilla_cost = _attempt_summary(vanilla_log)
hint_attempts, hint_in, hint_out, hint_cost = _attempt_summary(hint_log)
analysis_attempts, analysis_in, analysis_out, analysis_cost = _attempt_summary(analysis_log)
conflict_attempts, conflict_in, conflict_out, conflict_cost = _attempt_summary(conflict_log)

# Citation honesty check.
links_vanilla = _all_wikilinks(vanilla_page, vanilla_body)
all_resolved = all(link in available for link in links_vanilla)
all_inputs_cited = all(p in links_vanilla for p in source_paths)

# Did the conflict cell populate contradicts?
conflict_fm = conflict_page.model_dump(mode="json", exclude_none=True)
contradicts_populated = bool(conflict_fm.get("contradicts"))

receipts = f"""# Cross-source Synthesis Receipts

**Last verified:** {date.today().isoformat()}
**Generated by:** `notebooks/05_cross_source_synthesis.ipynb`
**Design ref:** `docs/marginalia-design.md` §7.1 (synthesis agent), §10A step 5, §7.3.1 (strict-schema retry).

## Run summary

| Variant | Target type | Attempts | Tokens (in/out) | Cost (USD) | Wall |
|---|---|---|---|---|---|
| Vanilla concept | concept | {vanilla_attempts} | {vanilla_in}/{vanilla_out} | ${vanilla_cost:.6f} | {vanilla_wall:.2f}s |
| Hinted concept (platform-engineering) | concept | {hint_attempts} | {hint_in}/{hint_out} | ${hint_cost:.6f} | {hint_wall:.2f}s |
| Target=Analysis | analysis | {analysis_attempts} | {analysis_in}/{analysis_out} | ${analysis_cost:.6f} | {analysis_wall:.2f}s |
| Contradictory pair | concept | {conflict_attempts} | {conflict_in}/{conflict_out} | ${conflict_cost:.6f} | {conflict_wall:.2f}s |

All runs use `claude-sonnet-4-6` (default for cross-source synthesis per design §7.2).

## Citation verification

Vanilla synthesis on the 3-source fixture set:

- All wikilinks resolve to known paths: **{all_resolved}**
- All input source paths cited: **{all_inputs_cited}**

The retry loop fed back any dangling/missing-citation errors as
`<validation_errors>` on the next attempt — the same channel as
Pydantic schema errors. See `engine.agents.synthesis.cross_source.verify_citations`.

## Hint behavior

| | Vanilla | Hinted |
|---|---|---|
| Title | {vanilla_page.title!r} | {hint_page.title!r} |
| Body length (chars) | {len(vanilla_body)} | {len(hint_body)} |

Hint is a soft steer — the same input sources produce a different
title and framing. Hint is not a hard constraint; the model is free
to honor or reframe.

## Contradiction handling

Contradictory pair (`apollo-q2.md` claiming Q2 launch, `apollo-q3.md`
claiming Q3 postponement):

- Final status: **{conflict_page.status.value}**
- `contradicts` field populated: **{contradicts_populated}**
- Attempts to first valid page: **{conflict_attempts}**

Sonnet 4.6 typically detects the conflict and either populates
`contradicts` on the new page or discusses the conflict in the body.
The §7.2 design table assigns "contradiction detection" to Opus 4.7
for production lint passes — NB 08 will receipt that. Cross-source
synthesis on Sonnet handles binary contradictions gracefully but
becomes less reliable on subtle, multi-step contradictions.

## Selected per design §7.1

| Decision | Choice | Rationale |
|---|---|---|
| Default synth model | Sonnet 4.6 | Balances reasoning depth with cost; §7.2 table. |
| Default target type | Concept | Most generic; Analysis/Decision available via `target_type`. |
| Citation enforcement | Inside retry loop | Same retry contract as Pydantic errors; mirrors §7.3.1. |
| `<thinking>` block | Required in prompt | Cross-source needs CoT; opposite of `ingest_synthesize.md` (which forbids it). |
| Source-page paths | Slug-derive from title | Same default in `derive_default_path()`; caller can override via `sources_paths`. |

## Caveats

- One run per variant; widen the sample before treating any cost or
  attempt count as canonical.
- The 3-source fixture is small. Real synthesis reads 10–20 pages
  (per §7.1) — Sonnet's reasoning at that scale needs separate
  receipts.
- The contradictory edge case uses two crystal-clear sources. Subtle
  contradictions (e.g., date drift, tone) are likely to slip past
  Sonnet — that's Opus territory per §7.2.
- No `existing_pages` were passed; production synthesis reads the
  index/affected entity neighborhood (NB 07 territory).

## Re-running

```python
from pathlib import Path
import frontmatter
from anthropic import Anthropic
from engine.agents.synthesis import synthesize_cross_source
from engine.models.pages import SourcePage
from engine.models.wiki_config import MarginaliaConfig

config = MarginaliaConfig.load(Path("notebooks/data/poc-wiki"))
client = Anthropic()
sources = [
    SourcePage.model_validate(frontmatter.load(p).metadata)
    for p in sorted(Path("notebooks/data/poc-wiki/sources").glob("*.md"))
    if not p.name.startswith("apollo-q")
]
page, body, log = await synthesize_cross_source(sources, config, client=client)
print(page.title, len(log))
```
"""

DECISIONS_PATH.write_text(receipts, encoding="utf-8")
print(f"wrote {DECISIONS_PATH.resolve()}  ({DECISIONS_PATH.stat().st_size} bytes)")


## What to extract

| Notebook artifact | Extracts to |
|---|---|
| `synthesize_cross_source()`, `verify_citations()`, `extract_wikilinks()`, `derive_default_path()` | `engine/agents/synthesis/cross_source.py` (extracted) |
| Versioned synthesis prompt | `engine/prompts/synthesis.md` (extracted) |
| Cell 10 receipts | `engine/decisions/synthesis-patterns.md` (extracted) |

**Notebook-only (intentionally not extracted):**
- The fixture loader (`_load_source_page`) — moves to `engine.models.wiki_io` if/when it's needed in production code.
- The citation visualizer (cell 5) — diagnostic for human inspection; `verify_citations` already returns the data.
- The cost summary table (cell 9) — receipts version lives in `synthesis-patterns.md`.

**Fixtures committed at `notebooks/data/poc-wiki/sources/`:**
- 3 auto-generated SourcePages (regenerate via `uv run python notebooks/_ops/build_source_pages.py`).
- 2 hand-curated contradictory SourcePages (`apollo-q2.md`, `apollo-q3.md`) — deliberately authored, do NOT regenerate.

**`CACHE_VERSION` discipline:** not triggered by NB 05 (no edits to `ingest_*.md`). The new `synthesis.md` starts at v1; future edits must bump in lockstep with `CACHE_VERSION` once it lands in NB 10.
